# Vocoder Comparison: Python mlx-audio vs Rust voicers

Same phonemes, same model weights, same voice — which produces intelligible audio?

If Python sounds fine and Rust doesn't, the bug is in our Rust vocoder (iSTFT, overlap-add, weight loading, etc).
If both sound bad, it's the model itself or an MLX framework issue.

In [1]:
import subprocess, json, os, tempfile
import numpy as np
import whisper

VOICERS_CLI = "./target/release/voicers-cli"

# Build release binary
result = subprocess.run(
    ["cargo", "build", "--release", "-p", "voicers-cli"], capture_output=True, text=True
)
print("voicers-cli:", "ready" if result.returncode == 0 else result.stderr[-200:])

# Load Whisper
whisper_model = whisper.load_model("base")
print("Whisper: ready")

FileNotFoundError: [Errno 2] No such file or directory: 'cargo'

In [2]:
import subprocess, json, os, tempfile
import numpy as np

# Full paths since notebook env may not have cargo in PATH
CARGO = os.path.expanduser("~/.cargo/bin/cargo")
VOICERS_CLI = os.path.abspath("./target/release/voicers-cli")

# Build release binary
result = subprocess.run(
    [CARGO, "build", "--release", "-p", "voicers-cli"], capture_output=True, text=True
)
print("voicers-cli:", "ready" if result.returncode == 0 else result.stderr[-200:])

import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

voicers-cli: ready
Whisper: ready


In [3]:
# Set up Python mlx-audio Kokoro pipeline
from mlx_audio.tts.models.kokoro.pipeline import KokoroPipeline
import soundfile as sf

pipeline = KokoroPipeline(lang_code="a")
print(f"Python mlx-audio pipeline ready, voice: af_heart")

ModuleNotFoundError: No module named 'misaki'

In [1]:
import subprocess, json, os, tempfile
import numpy as np

CARGO = os.path.expanduser("~/.cargo/bin/cargo")
VOICERS_CLI = os.path.abspath("./target/release/voicers-cli")

# Build
result = subprocess.run(
    [CARGO, "build", "--release", "-p", "voicers-cli"], capture_output=True, text=True
)
print("voicers-cli:", "ready" if result.returncode == 0 else result.stderr[-200:])

import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

voicers-cli: ready
Whisper: ready


In [2]:
# Set up Python mlx-audio Kokoro pipeline
from mlx_audio.tts.models.kokoro.pipeline import KokoroPipeline
import soundfile as sf

pipeline = KokoroPipeline(lang_code="a")
print(f"Python mlx-audio Kokoro pipeline ready")

TypeError: KokoroPipeline.__init__() missing 2 required positional arguments: 'model' and 'repo_id'

In [3]:
# Use mlx-audio's generate CLI and our voicers CLI to produce audio from the same text
# Then transcribe both with Whisper

import soundfile as sf


def generate_python_audio(text, voice="af_heart", output_path=None):
    """Generate audio using Python mlx-audio Kokoro."""
    if output_path is None:
        output_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [
            "python3",
            "-m",
            "mlx_audio.tts.generate",
            "--model",
            "prince-canuma/Kokoro-82M",
            "--text",
            text,
            "--voice",
            voice,
            "--output",
            output_path,
        ],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if r.returncode != 0:
        print(f"Python gen failed: {r.stderr[-300:]}")
        return None
    return output_path


def generate_rust_audio(text, voice="af_heart", output_path=None):
    """Generate audio using Rust voicers CLI."""
    if output_path is None:
        output_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", output_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if r.returncode != 0:
        print(f"Rust gen failed: {r.stderr[-300:]}")
        return None
    # Extract phonemes from stderr
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return output_path, phonemes.strip()


def transcribe(wav_path):
    """Transcribe audio with Whisper."""
    result = whisper_model.transcribe(wav_path, language="en")
    return result["text"].strip()


# Quick smoke test
print("Testing Python mlx-audio...")
py_path = generate_python_audio("Hello world")
print(f"  Generated: {py_path}")
if py_path:
    py_whisper = transcribe(py_path)
    print(f"  Whisper heard: {py_whisper}")
    os.unlink(py_path)

Testing Python mlx-audio...
Python gen failed: /Applications/Xcode.app/Contents/Developer/usr/bin/python3: E
rror while finding module specification for 'mlx_audio.tts.generate' (ModuleNotF
oundError: No module named 'mlx_audio')

  Generated: None


In [4]:
# Generate Python audio directly in this notebook since we have mlx-audio installed here
import mlx.core as mx
from mlx_audio.tts.models.kokoro.kokoro import (
    Model,
    ModelConfig,
    sanitize,
    sanitize_lstm_weights,
)
from mlx_audio.tts.models.kokoro.voice import load_voice_tensor
from huggingface_hub import snapshot_download
import mlx.nn as nn
import json
from pathlib import Path

# Load model
repo_id = "prince-canuma/Kokoro-82M"
model_path = Path(snapshot_download(repo_id))
print(f"Model path: {model_path}")

with open(model_path / "config.json") as f:
    config_data = json.load(f)

config = ModelConfig.from_dict(config_data)
model = Model(config)

# Load weights
weights = mx.load(str(model_path / "model.safetensors"))
sanitized = sanitize(weights, config)
model.load_weights(list(sanitized.items()), strict=False)
mx.eval(model.parameters())
print("Model loaded")

# Load voice
voice = load_voice_tensor(repo_id, "af_heart")
print(f"Voice loaded: {voice.shape}")

ImportError: cannot import name 'sanitize' from 'mlx_audio.tts.models.kokoro.kokoro' (/Users/kylekelley/Library/Caches/runt/inline-envs/578aa62ad89c9c27/lib/python3.12/site-packages/mlx_audio/tts/models/kokoro/kokoro.py)

In [5]:
# Let's see what mlx-audio's Model exposes — the installed version may differ from colombo-v2 source
from mlx_audio.tts.models.kokoro.kokoro import Model, ModelConfig

print(dir(Model))

# Use the high-level generate API instead of manually loading
# First, let's see what the Model.generate method signature looks like
import inspect

sig = inspect.signature(Model.generate)
print(f"\nModel.generate signature: {sig}")

['Output', 'REPO_ID', '__annotations__', '__call__', '__class__', '__class_getit
em__', '__contains__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__d
oc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__g
etitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__'
, '__ior__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__',
'__new__', '__or__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__',
'__ror__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshoo
k__', '__weakref__', '_extra_repr', '_get_pipeline', '_set_training_mode', '_val
idate_keys', 'apply', 'apply_to_modules', 'children', 'clear', 'copy', 'eval', '
filter_and_map', 'freeze', 'fromkeys', 'generate', 'get', 'is_module', 'items',
'keys', 'leaf_modules', 'load_weights', 'modules', 'named_modules', 'parameters'
, 'pop', 'popitem', 'sample_rate', 'sanitize', 'save_weights', 'set_dtype', 'set
default', 'state', 'train', 'tr

In [6]:
# Use the high-level API — Model handles its own loading
from mlx_audio.tts.utils import load as load_tts

py_model, py_processor = load_tts("prince-canuma/Kokoro-82M")
print(f"Python model loaded, sample_rate={py_model.sample_rate}")

Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

ValueError: too many values to unpack (expected 2)

In [7]:
# Check what load returns
from mlx_audio.tts.utils import load as load_tts

result = load_tts("prince-canuma/Kokoro-82M")
print(type(result))
if isinstance(result, tuple):
    print(f"Tuple length: {len(result)}")
    for i, r in enumerate(result):
        print(f"  [{i}] {type(r)}")
else:
    py_model = result
    print(f"Single object: {type(py_model)}")

Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

<class 'mlx_audio.tts.models.kokoro.kokoro.Model'>
Single object: <class 'mlx_audio.tts.models.kokoro.kokoro.Model'>


In [8]:
# Now we have the Python model. Let's generate audio!
import soundfile as sf

py_model = result  # from previous cell


def python_generate(text, voice="af_heart"):
    """Generate audio using Python mlx-audio and return wav path."""
    wav_path = tempfile.mktemp(suffix=".wav")
    for gen_result in py_model.generate(text, voice=voice, lang_code="a"):
        audio = np.array(gen_result.audio)
        sf.write(wav_path, audio, gen_result.sample_rate)
        return wav_path  # just take first segment
    return None


def rust_generate(text, voice="af_heart"):
    """Generate audio using Rust voicers and return (wav_path, phonemes)."""
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


# Smoke test
print("=== Python mlx-audio ===")
py_wav = python_generate("Hello world")
py_text = transcribe(py_wav)
print(f"  Whisper heard: {py_text}")

print("\n=== Rust voicers ===")
rust_wav, rust_ph = rust_generate("Hello world")
rust_text = transcribe(rust_wav)
print(f"  Phonemes: {rust_ph}")
print(f"  Whisper heard: {rust_text}")

# Cleanup
os.unlink(py_wav)
os.unlink(rust_wav)

=== Python mlx-audio ===
Creating new KokoroPipeline for language: a


/Users/kylekelley/Library/Caches/runt/inline-envs/578aa62ad89c9c27/bin/python: N
o module named pip


SystemExit: 1

/Users/kylekelley/Library/Caches/runt/inline-envs/578aa62ad89c9c27/lib/python3.1
2/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use
 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import subprocess, os, tempfile
import numpy as np

CARGO = os.path.expanduser("~/.cargo/bin/cargo")
VOICERS_CLI = os.path.abspath("./target/release/voicers-cli")

# Verify spacy model is available
import spacy

nlp = spacy.load("en_core_web_sm")
print("spaCy en_core_web_sm: ready")

# Load Whisper
import whisper

whisper_model = whisper.load_model("base")
print("Whisper: ready")

# Load Python mlx-audio model
from mlx_audio.tts.utils import load as load_tts

py_model = load_tts("prince-canuma/Kokoro-82M")
print(f"Python Kokoro: ready (sample_rate={py_model.sample_rate})")

spaCy en_core_web_sm: ready
Whisper: ready


Fetching 63 files:   0%|          | 0/63 [00:00<?, ?it/s]

Python Kokoro: ready (sample_rate=24000)


In [2]:
import soundfile as sf


def python_generate(text, voice="af_heart"):
    """Generate audio using Python mlx-audio Kokoro."""
    wav_path = tempfile.mktemp(suffix=".wav")
    for gen_result in py_model.generate(text, voice=voice, lang_code="a"):
        audio = np.array(gen_result.audio)
        sf.write(wav_path, audio, gen_result.sample_rate)
        return wav_path
    return None


def rust_generate(text, voice="af_heart"):
    """Generate audio using Rust voicers."""
    wav_path = tempfile.mktemp(suffix=".wav")
    r = subprocess.run(
        [VOICERS_CLI, "--text", text, "--voice", voice, "--output", wav_path],
        capture_output=True,
        text=True,
        timeout=120,
    )
    phonemes = ""
    for line in r.stderr.splitlines():
        if line.strip().startswith("chunk"):
            phonemes += line.split(":", 1)[1].strip() + " "
    return wav_path, phonemes.strip()


def transcribe(wav_path):
    result = whisper_model.transcribe(wav_path, language="en")
    return result["text"].strip()


# Smoke test both
print("=== Python mlx-audio ===")
py_wav = python_generate("Hello world")
print(f"  Whisper heard: {transcribe(py_wav)}")
os.unlink(py_wav)

print("\n=== Rust voicers ===")
rust_wav, rust_ph = rust_generate("Hello world")
print(f"  Phonemes: {rust_ph}")
print(f"  Whisper heard: {transcribe(rust_wav)}")
os.unlink(rust_wav)

=== Python mlx-audio ===
Creating new KokoroPipeline for language: a


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


=== Python mlx-audio ===
Creating new KokoroPipeline for language: a
  Whisper heard: Hello world.

=== Rust voicers ===
  Phonemes: həlˈO wˈɜɹld


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


=== Python mlx-audio ===
Creating new KokoroPipeline for language: a
  Whisper heard: Hello world.

=== Rust voicers ===
  Phonemes: həlˈO wˈɜɹld
  Whisper heard: Hello, where are you?


In [3]:
# Full A/B comparison
test_phrases = [
    "Hello world",
    "Good morning",
    "The quick brown fox",
    "How are you doing today?",
    "She sells seashells by the seashore.",
    "I read a book yesterday.",
    "The dog is running in the park.",
]

print(f"{'Input':<40} {'Python Whisper':<30} {'Rust Whisper':<30}")
print("=" * 100)

for phrase in test_phrases:
    # Python
    py_wav = python_generate(phrase)
    py_heard = transcribe(py_wav) if py_wav else "FAILED"
    os.unlink(py_wav) if py_wav else None

    # Rust
    rust_wav, rust_ph = rust_generate(phrase)
    rust_heard = transcribe(rust_wav)
    os.unlink(rust_wav)

    py_ok = "OK" if phrase.lower().rstrip(".,!?") in py_heard.lower() else ""
    rust_ok = "OK" if phrase.lower().rstrip(".,!?") in rust_heard.lower() else ""

    print(f"{phrase:<40} {py_heard[:28]:<30} {rust_heard[:28]:<30}")

Input                                    Python Whisper                 Rust Whi
sper


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey


...

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey
The quick brown fox                      The quick brown fox.           The cool
, very evil!
How are you doing today?                 How are you doing today?       How art
you doing to say
She sells seashells by the seashore.     She sells seashells by the s   She sell
s a sea shells by th


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey
The quick brown fox                      The quick brown fox.           The cool
, very evil!
How are you doing today?                 How are you doing today?       How art
you doing to say
She sells seashells by the seashore.     She sells seashells by the s   She sell
s a sea shells by th
I read a book yesterday.                 I read a book yesterday.       I read a
 book, yes, to Herds


/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/kylekelley/Library/Caches/runt/inline-envs/da6ba3ffe24a2e47/lib/python3.1
2/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on
 CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; u

Input                                    Python Whisper                 Rust Whi
sper
Hello world                              Hello world.                   Hello, w
here? Holds that.
Good morning                             Good morning.                  Good mor
ning, hey
The quick brown fox                      The quick brown fox.           The cool
, very evil!
How are you doing today?                 How are you doing today?       How art
you doing to say
She sells seashells by the seashore.     She sells seashells by the s   She sell
s a sea shells by th
I read a book yesterday.                 I read a book yesterday.       I read a
 book, yes, to Herds
The dog is running in the park.          The dog is running in the pa   The Dizd
og is as running int


## Results: Python mlx-audio vs Rust voicers

| Input | Python (Whisper) | Rust (Whisper) |
|-------|-----------------|----------------|
| Hello world | Hello world. | Hello, where? Holds that. |
| Good morning | Good morning. | Good morning, hey |
| The quick brown fox | The quick brown fox. | The cool, very evil! |
| How are you doing today? | How are you doing today? | How art you doing to say |
| She sells seashells... | She sells seashells by the s... | She sells a sea shells by th... |
| I read a book yesterday. | I read a book yesterday. | I read a book, yes, to Herds |
| The dog is running... | The dog is running in the pa... | The Dizdog is as running int... |

**Python: 7/7 correct. Rust: 0/7 correct.**

The Python mlx-audio pipeline produces perfect audio from the same model. Our Rust vocoder is the problem. The issue is somewhere in:
1. **iSTFT reconstruction** (overlap-add, window normalization)
2. **Weight loading/sanitization** (some weights may still be misaligned)
3. **Forward pass numerics** (intermediate computation differences)

Next: dump intermediate tensors from both Python and Rust to find where they diverge.